# 1 SETUP E VERIFICAÇÃO

In [3]:
from pathlib import Path
import torch
from ultralytics import YOLO

# Detecta a pasta do projeto automaticamente (sobe 1 nível do notebook)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset" / "RDD_SPLIT"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)

CLASSES = {0: "D00", 1: "D10", 2: "D20", 3: "D40", 4: "D43"}

assert DATASET_DIR.exists(), f"Dataset não encontrado: {DATASET_DIR}"
assert torch.backends.mps.is_available(), "MPS não disponível"

device = "mps"
print(f"✅ Project: {PROJECT_ROOT}")
print(f"✅ Device: {device}")
print(f"✅ Dataset: {DATASET_DIR}")
for subset in ["train", "val", "test"]:
    n = sum(1 for _ in (DATASET_DIR / subset / "images").iterdir())
    print(f"   {subset}: {n:,} imagens")

✅ Project: /Users/sergiomendes/Documents/POS_IA/road-damage-yolo
✅ Device: mps
✅ Dataset: /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/dataset/RDD_SPLIT
   train: 26,869 imagens
   val: 5,758 imagens
   test: 5,758 imagens


# 2. CRIAR `data.yml`

In [4]:
import yaml

DATA_YAML = PROJECT_ROOT / "data.yaml"

content = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(CLASSES),
    "names": [CLASSES[i] for i in sorted(CLASSES.keys())],
}

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(content, f, sort_keys=False)

print(f"✅ {DATA_YAML}\n")
with open(DATA_YAML) as f:
    print(f.read())

✅ /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/data.yaml

path: /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/dataset/RDD_SPLIT
train: train/images
val: val/images
test: test/images
nc: 5
names:
- D00
- D10
- D20
- D40
- D43



# 3. TREINO

In [5]:
model = YOLO("yolo11s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=8,
    patience=10,
    device="mps",
    workers=4,
    cache=False,
    amp=False,           # MPS tem bugs com AMP em algumas versões — desligado por segurança
    seed=42,
    project=str(OUTPUTS_DIR),
    name="yolo11s_rdd2022",
    exist_ok=False,
    plots=True,
    verbose=True,
)

print("\n✅ Treino concluído")
print(f"Pesos salvos em: {OUTPUTS_DIR / 'yolo11s_rdd2022' / 'weights' / 'best.pt'}")

Ultralytics 8.4.51 🚀 Python-3.13.2 torch-2.12.0 MPS (Apple M3)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/sergiomendes/Documents/POS_IA/road-damage-yolo/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_rdd2022, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [6]:
import pandas as pd

run_dir = OUTPUTS_DIR / "yolo11s_rdd2022"
results_csv = run_dir / "results.csv"

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print(f"Total de épocas treinadas: {len(df)}\n")
print("Últimas 5 épocas:")
cols = ['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 
        'metrics/precision(B)', 'metrics/recall(B)']
cols_existing = [c for c in cols if c in df.columns]
print(df[cols_existing].tail().to_string(index=False))

print(f"\nMelhor mAP@0.5: {df['metrics/mAP50(B)'].max():.4f}")
print(f"Melhor época: {df['metrics/mAP50(B)'].idxmax() + 1}")

Total de épocas treinadas: 30

Últimas 5 épocas:
 epoch  metrics/mAP50(B)  metrics/mAP50-95(B)  metrics/precision(B)  metrics/recall(B)
    26           0.59242              0.31987               0.64210            0.53986
    27           0.59454              0.32185               0.63920            0.54493
    28           0.59639              0.32343               0.63910            0.54771
    29           0.59820              0.32415               0.63426            0.55374
    30           0.59944              0.32492               0.64222            0.55022

Melhor mAP@0.5: 0.5994
Melhor época: 30


# 4. Avaliação no test set

In [7]:
model = YOLO(str(OUTPUTS_DIR / "yolo11s_rdd2022" / "weights" / "best.pt"))

test_results = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device="mps",
    plots=True,
    save_json=True,
    project=str(OUTPUTS_DIR),
    name="test_eval",
    exist_ok=True,
)

print("\n" + "="*50)
print("MÉTRICAS NO TEST SET")
print("="*50)
print(f"mAP@0.5:       {test_results.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {test_results.box.map:.4f}")
print(f"Precision:     {test_results.box.mp:.4f}")
print(f"Recall:        {test_results.box.mr:.4f}")

print("\n" + "="*50)
print("MÉTRICAS POR CLASSE")
print("="*50)
class_names = [CLASSES[i] for i in sorted(CLASSES.keys())]
print(f"{'Classe':<8} {'Precision':<12} {'Recall':<10} {'mAP@0.5':<10} {'mAP@0.5:0.95':<12}")
for i, name in enumerate(class_names):
    p = test_results.box.p[i]
    r = test_results.box.r[i]
    ap50 = test_results.box.ap50[i]
    ap = test_results.box.ap[i]
    print(f"{name:<8} {p:<12.4f} {r:<10.4f} {ap50:<10.4f} {ap:<12.4f}")

Ultralytics 8.4.51 🚀 Python-3.13.2 torch-2.12.0 MPS (Apple M3)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 320.0±76.7 MB/s, size: 72.8 KB)
val: Scanning /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/dataset/RDD_SPLIT/test/labels... 5758 images, 1790 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5758/5758 3.4Kit/s 1.7s0.1s
val: New cache created: /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/dataset/RDD_SPLIT/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 720/720 2.4it/s 4:56<0.4s
                   all       5758       9675      0.641       0.56      0.594      0.317
                   D00       2080       3925      0.621      0.536      0.563      0.312
                   D10       1118       1675       0.59      0.531       0.54      0.265
                   D20       1214       1537      0.693      0.597 

# 5. Inferência em 20 Imagens

In [8]:
import random
import shutil

random.seed(42)

# Carrega o modelo
model = YOLO(str(OUTPUTS_DIR / "yolo11s_rdd2022" / "weights" / "best.pt"))

# Pasta pra salvar exemplos
EXAMPLES_DIR = OUTPUTS_DIR / "examples"
if EXAMPLES_DIR.exists():
    shutil.rmtree(EXAMPLES_DIR)
EXAMPLES_DIR.mkdir(parents=True)

# Pega 20 imagens aleatórias do test set
test_images = list((DATASET_DIR / "test" / "images").iterdir())
test_images = [f for f in test_images if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]

# Filtra só imagens que têm anotações (mais interessantes pra demo)
test_images_with_labels = []
for img in test_images:
    lbl = DATASET_DIR / "test" / "labels" / f"{img.stem}.txt"
    if lbl.exists() and lbl.stat().st_size > 0:
        test_images_with_labels.append(img)

selected = random.sample(test_images_with_labels, 20)

print(f"Rodando inferência em 20 imagens...")
results = model.predict(
    source=[str(f) for f in selected],
    conf=0.25,
    imgsz=640,
    device="mps",
    save=True,
    project=str(OUTPUTS_DIR),
    name="examples_predictions",
    exist_ok=True,
    verbose=False,
)

# Move arquivos pra pasta examples/ com nomes mais limpos
pred_dir = OUTPUTS_DIR / "examples_predictions"
for i, src in enumerate(sorted(pred_dir.glob("*.jpg")), 1):
    dst = EXAMPLES_DIR / f"example_{i:02d}_{src.stem}.jpg"
    shutil.copy(src, dst)

# Estatísticas das detecções
total_detections = sum(len(r.boxes) for r in results)
print(f"\n✅ {len(selected)} imagens processadas")
print(f"   Total de detecções: {total_detections}")
print(f"   Média por imagem: {total_detections / len(selected):.1f}")
print(f"\n📁 Salvos em: {EXAMPLES_DIR}")

Rodando inferência em 20 imagens...
Results saved to /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/outputs/examples_predictions

✅ 20 imagens processadas
   Total de detecções: 44
   Média por imagem: 2.2

📁 Salvos em: /Users/sergiomendes/Documents/POS_IA/road-damage-yolo/outputs/examples
